**Импорт необходимых библиотек**

In [6]:
import open3d as o3d
import numpy as np
import random
import plotly.graph_objects as go
print("✅ Библиотеки загружены")

✅ Библиотеки загружены


**Пример из библиотеки Open3D**

In [2]:
sample = o3d.data.PLYPointCloud()
pcd = o3d.io.read_point_cloud(sample.path)

print(f"Загружено точек: {len(pcd.points)}")

pcd = pcd.voxel_down_sample(voxel_size=0.03)
points = np.asarray(pcd.points)
print(f"После даунсемплинга: {len(points)} точек")

[Open3D INFO] Downloading https://github.com/isl-org/open3d_downloads/releases/download/20220201-data/fragment.ply
[Open3D INFO] Downloaded to C:\Users\Andrei/open3d_data/download/PLYPointCloud/fragment.ply
Загружено точек: 196133
После даунсемплинга: 12369 точек


**Реализация RANSAC**

In [3]:
def ransac_plane_segmentation(points, max_iter=1000, dist_threshold=0.025):
    n_points = len(points)
    best_inlier_count = 0
    best_plane = None
    best_inlier_mask = None

    for _ in range(max_iter):
        # 1. Выбираем 3 случайные точки
        sample_idx = random.sample(range(n_points), 3)
        p1, p2, p3 = points[sample_idx]

        # 2. Строим уравнение плоскости ax + by + cz + d = 0
        v1 = p2 - p1
        v2 = p3 - p1
        normal = np.cross(v1, v2)
        norm = np.linalg.norm(normal)
        if norm < 1e-6:
            continue
        normal = normal / norm
        a, b, c = normal
        d = -np.dot(normal, p1)

        # 3. Расстояние от всех точек до плоскости
        distances = np.abs(points @ np.array([a, b, c]) + d)

        # 4. Инлайеры
        inlier_mask = distances < dist_threshold
        inlier_count = np.sum(inlier_mask)

        # 5. Сохраняем лучшую модель
        if inlier_count > best_inlier_count:
            best_inlier_count = inlier_count
            best_plane = (a, b, c, d)
            best_inlier_mask = inlier_mask.copy()

    print(f"✅ Лучшая плоскость найдена!")
    print(f"   Инлайеров: {best_inlier_count} из {n_points} ({best_inlier_count/n_points*100:.1f}%)")
    print(f"   Параметры плоскости: a={best_plane[0]:.4f}, b={best_plane[1]:.4f}, c={best_plane[2]:.4f}, d={best_plane[3]:.4f}")
    return best_plane, best_inlier_mask

plane_params, inlier_mask = ransac_plane_segmentation(points, max_iter=1000, dist_threshold=0.025)

✅ Лучшая плоскость найдена!
   Инлайеров: 3015 из 12369 (24.4%)
   Параметры плоскости: a=-0.0047, b=-0.9999, c=-0.0144, d=2.4244


**Результат**

In [14]:
colors = np.full((len(points), 3), [0.9, 0.15, 0.15], dtype=float)   # красный по умолчанию
colors[inlier_mask] = [0.1, 0.75, 0.2]                                 # зелёный — пол

if len(points) > 150000:
    step = len(points) // 150000
    points_plot = points[::step]
    colors_plot = colors[::step]
    print(f"Отрисовываем {len(points_plot):,} точек (прорежено)")
else:
    points_plot = points
    colors_plot = colors

fig = go.Figure()

fig.add_trace(go.Scatter3d(
    x=points_plot[:, 0],
    y=points_plot[:, 1],
    z=points_plot[:, 2],
    mode='markers',
    marker=dict(
        size=2.5,
        color=colors_plot,
        opacity=0.9
    ),
    name='Облако точек'
))

# Настройки вида
fig.update_layout(
    title="RANSAC → Пол (зелёный) / Препятствия (красный)",
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z',
        aspectmode='data',
        camera=dict(
            eye=dict(x=1.8, y=-1.8, z=1.2)
        )
    ),
    width=1000,
    height=800,
    margin=dict(l=0, r=0, b=0, t=50),
    showlegend=False
)

fig.show()